In [1]:
import ast
import string

import pandas as pd

DATASET_PATH = "../../data/out/distillation/hendrycks_math_distilled_extra_detailed_raw_deepseek_v4_flash_regenerate_incorrect_w_large.parquet"

df = pd.read_parquet(DATASET_PATH)
print(f"Path: {DATASET_PATH}")
print(f"Shape: {df.shape}")
df.head(2)

Path: ../../data/out/distillation/hendrycks_math_distilled_extra_detailed_raw_deepseek_v4_flash_regenerate_incorrect_w_large.parquet
Shape: (7500, 7)


,question,answer,answer_w_steps,question_id,distill_ans_correct,distill_reasoning,distill_answer
0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",0,"For the piecewise function to be continuous, t...",0,True,We need answer solve with maximal detail. Need...,Let the piecewise function be\n\n\[\nf(x)=\n\b...
1,A rectangular band formation is a formation wi...,98,Let $x$ be the number of band members in each ...,1,True,We need solve problem. Need produce detailed s...,Let \(N\) be the total number of band members....


## Accuracy


In [2]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 0.9360 (7020/7500)


## Missing distilled reasoning traces


In [3]:
missing_nan = int(df["distill_answer"].isna().sum())
empty_mask = df["distill_answer"].fillna("").str.strip() == ""
missing_or_empty = int(empty_mask.sum())
print(f"NaN distill_reasoning:            {missing_nan}")
print(f"Missing or empty distill_reasoning: {missing_or_empty}")

NaN distill_reasoning:            0
Missing or empty distill_reasoning: 12


## Reasoning trace length distribution (chars)


In [4]:
existing_reasoning = df.loc[~empty_mask, "distill_answer"]
lengths = existing_reasoning.str.len()
print(lengths.describe())
print(f"P60: {lengths.quantile(0.6)}")
print(f"P70: {lengths.quantile(0.7)}")
print(f"P80: {lengths.quantile(0.8)}")
print(f"P85: {lengths.quantile(0.85)}")
print(f"P90: {lengths.quantile(0.9)}")
print(f"P92: {lengths.quantile(0.92)}")
print(f"P95: {lengths.quantile(0.95)}")
print(f"P99: {lengths.quantile(0.99)}")

count     7488.000000
mean      2738.487447
std       1151.836161
min        551.000000
25%       1913.000000
50%       2514.000000
75%       3310.250000
max      10792.000000
Name: distill_answer, dtype: float64
P60: 2790.0
P70: 3103.0
P80: 3566.0
P85: 3875.0
P90: 4291.900000000001
P92: 4511.04
P95: 4939.65
P99: 6451.910000000001


In [5]:
# corrected_empty_mask = df["corrected_reasoning"].fillna("").str.strip() == ""
# existing_corrected = df.loc[~corrected_empty_mask, "corrected_reasoning"]
# lengths = existing_corrected.str.len()
# print(lengths.describe())
# print(f"P60: {lengths.quantile(0.6)}")
# print(f"P70: {lengths.quantile(0.7)}")
# print(f"P80: {lengths.quantile(0.8)}")
# print(f"P85: {lengths.quantile(0.85)}")
# print(f"P90: {lengths.quantile(0.9)}")
# print(f"P92: {lengths.quantile(0.92)}")
# print(f"P95: {lengths.quantile(0.95)}")
# print(f"P99: {lengths.quantile(0.99)}")

## Reasoning trace length distribution (tokens)

Token counts per reasoning trace under each of the three model tokenizers we use: `llama3b` (`meta-llama/Llama-3.2-3B-Instruct`), `phi4mini` (`microsoft/Phi-4-mini-instruct`), and `qwen3b` (`Qwen/Qwen2.5-3B-Instruct`).

In [6]:
# from tqdm.auto import tqdm
# from transformers import AutoTokenizer

# MODEL_TOKENIZERS = {
#     "llama3b": "meta-llama/Llama-3.2-3B-Instruct",
#     "phi4mini": "microsoft/Phi-4-mini-instruct",
#     "qwen3b": "Qwen/Qwen2.5-3B-Instruct",
# }

# reasoning_texts = existing_reasoning.tolist()

# for name, model_id in MODEL_TOKENIZERS.items():
#     tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
#     token_lengths = pd.Series(
#         [len(tokenizer(text, add_special_tokens=False)["input_ids"]) for text in tqdm(reasoning_texts, desc=name)],
#         index=existing_reasoning.index,
#     )
#     print(f"=== {name} ({model_id}) ===")
#     print(token_lengths.describe())
#     print(f"P60: {token_lengths.quantile(0.6)}")
#     print(f"P70: {token_lengths.quantile(0.7)}")
#     print(f"P80: {token_lengths.quantile(0.8)}")
#     print(f"P85: {token_lengths.quantile(0.85)}")
#     print(f"P90: {token_lengths.quantile(0.9)}")
#     print(f"P92: {token_lengths.quantile(0.92)}")
#     print(f"P95: {token_lengths.quantile(0.95)}")
#     print(f"P99: {token_lengths.quantile(0.99)}")
#     print()

## Invalid answers

An answer is valid iff, after stripping/lowercasing, it is one of the option letters `a`, `b`, ... up to the row's number of options.


In [5]:
LETTERS = list(string.ascii_lowercase)


def n_options(opts):
    if isinstance(opts, str):
        try:
            return len(ast.literal_eval(opts))
        except Exception:
            return 0
    try:
        return len(opts)
    except TypeError:
        return 0


def is_valid_answer(ans, n_opts):
    if not isinstance(ans, str):
        return False
    return ans.strip().lower() in set(LETTERS[:n_opts])


# n_opts_series = df["options"].apply(n_options)
valid_mask = df["distill_ans_correct"]
n_invalid = int((~valid_mask).sum())
print(f"Invalid distill_answer count: {n_invalid} ({n_invalid / n:.2%})")

Invalid distill_answer count: 480 (6.40%)


### All invalid answers


In [6]:
invalid_answers = df.loc[~valid_mask, "distill_answer"]
for idx, ans in invalid_answers.items():
    print(f"--- row {idx} ---")
    print(repr(ans))

--- row 12 ---
'We need to determine how many euros Berengere must contribute so that together she and Emily can pay for a cake costing \\(6\\) euros.\n\n**Step 1: State what is given.**\n\nThe cake costs \\(6\\) euros.\n\nEmily has an American five-dollar bill, so Emily has\n\n\\[\n5 \\text{ USD}.\n\\]\n\nThe exchange rate is\n\n\\[\n1 \\text{ euro} = 1.25 \\text{ USD}.\n\\]\n\nThis means that one euro has the same value as \\(1.25\\) American dollars.\n\n**Step 2: Convert Emily’s \\(5\\) dollars into euros.**\n\nWe want to know how many euros are equivalent to \\(5\\) USD.\n\nSince\n\n\\[\n1 \\text{ euro} = 1.25 \\text{ USD},\n\\]\n\nwe can find how many euros correspond to \\(1\\) USD.\n\nIf \\(1.25\\) USD is worth \\(1\\) euro, then \\(1\\) USD is worth\n\n\\[\n\\frac{1}{1.25} \\text{ euros}.\n\\]\n\nNow write \\(1.25\\) as a fraction:\n\n\\[\n1.25 = \\frac{125}{100} = \\frac{5}{4}.\n\\]\n\nTherefore,\n\n\\[\n\\frac{1}{1.25} = \\frac{1}{\\frac{5}{4}}.\n\\]\n\nDividing by a fraction

### 10 shortest reasoning traces


In [7]:
shortest = existing_reasoning.str.len().nsmallest(30)
for idx, length in shortest.items():
    print(f"--- row {idx} (len={length}) ---")
    print(df.loc[idx, "distill_answer"])
    print()

--- row 630 (len=551) ---
The function is defined by the rule that for every real number input, the output is \(2\). In symbols, for every real number \(u\),

\[
f(u)=2.
\]

We want to find \(f(x+2)\). The expression \(x+2\) is being used as the input to the function \(f\).

Because \(x\) is a real number, and the sum of two real numbers is always a real number, the quantity \(x+2\) is also a real number.

Therefore, the rule \(f(u)=2\) applies with

\[
u=x+2.
\]

Substituting \(u=x+2\) into the rule gives

\[
f(x+2)=2.
\]

So the value of \(f(x+2)\) is \(2\).

Answer: 2

--- row 7356 (len=630) ---
The determinant of a \(2 \times 2\) matrix is computed using the formula

\[
\begin{vmatrix} a & b \\ c & d \end{vmatrix}=ad-bc.
\]

For the given determinant

\[
\begin{vmatrix} -5 & 3 \\ 4 & -4 \end{vmatrix},
\]

we identify the entries as

\[
a=-5,\qquad b=3,\qquad c=4,\qquad d=-4.
\]

Now apply the formula \(ad-bc\):

\[
\begin{vmatrix} -5 & 3 \\ 4 & -4 \end{vmatrix}
=(-5)(-4)-(3)(4).
\]

## Reset distill columns for invalid answers

Sets `distill_reasoning` and `distill_answer` to `""` and `distill_ans_correct` to `False` for rows where the answer is invalid, then writes back to `DATASET_PATH`.


In [10]:
invalid_idx = df.index[~valid_mask]
df.loc[invalid_idx, "distill_reasoning"] = ""
df.loc[invalid_idx, "distill_answer"] = ""
df.loc[invalid_idx, "distill_ans_correct"] = False

df.to_parquet(DATASET_PATH, index=False)
print(f"Reset {len(invalid_idx)} rows and wrote {DATASET_PATH}")

KeyboardInterrupt: 

## Reset reasoning traces shorter than 10 chars

Sets `distill_reasoning` to `""` for rows where the existing trace is shorter than 10 characters, then writes back to `DATASET_PATH`.

In [ ]:
# short_mask = df["distill_reasoning"].fillna("").str.len() < 200
# short_idx = df.index[short_mask]
# df.loc[short_idx, "distill_reasoning"] = ""
# df.loc[short_idx, "distill_answer"] = ""
# df.loc[short_idx, "distill_ans_correct"] = False
# print(f"distill_reasoning rows under {200} chars: {len(short_idx)}")

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")

In [12]:
max_reasoning_chars = 8192
long_mask = df["distill_reasoning"].str.len() > max_reasoning_chars
print(f"distill_reasoning rows over {max_reasoning_chars} chars: {int(long_mask.sum())}")
df.loc[long_mask, "distill_reasoning"] = (
    df.loc[long_mask, "distill_reasoning"].str.slice(stop=max_reasoning_chars)
)
# df.loc[long_mask, "distill_reasoning"] = ""
# df.loc[long_mask, "distill_answer"] = ""
# df.loc[long_mask, "distill_ans_correct"] = False

distill_reasoning rows over 8192 chars: 10


In [ ]:
missing_reasoning_mask = df["distill_reasoning"].fillna("").eq("")
print(f"Dropping {int(missing_reasoning_mask.sum())} rows with NA or empty distill_reasoning")
df = df.loc[~missing_reasoning_mask].copy()

Dropping 83 rows with NA or empty distill_reasoning


In [ ]:
max_corrected_reasoning_chars = 1444
max_combined_reasoning_chars = 8192

# df["corrected_reasoning"] = df["corrected_reasoning"].str.slice(
#     stop=max_corrected_reasoning_chars
# )
has_corrected_reasoning = df["corrected_reasoning"].fillna("").str.strip().ne("")
df.loc[has_corrected_reasoning, "distill_reasoning"] = [
    (reasoning if isinstance(reasoning, str) else "")[:
        max_combined_reasoning_chars - len(corrected)
    ] + corrected
    for reasoning, corrected in zip(
        df.loc[has_corrected_reasoning, "distill_reasoning"],
        df.loc[has_corrected_reasoning, "corrected_reasoning"],
    )
]

In [8]:
df["distill_reasoning"] = df["distill_answer"]

In [9]:
from core.utils.cot_answers import extract_math_answer

df["distill_answer"] = [extract_math_answer(assistant_response, []) for assistant_response in df["distill_answer"]]


In [10]:
df.head()


,question,answer,answer_w_steps,question_id,distill_ans_correct,distill_reasoning,distill_answer
0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",0,"For the piecewise function to be continuous, t...",0,True,Let the piecewise function be\n\n\[\nf(x)=\n\b...,0
1,A rectangular band formation is a formation wi...,98,Let $x$ be the number of band members in each ...,1,True,Let \(N\) be the total number of band members....,98
2,What is the degree of the polynomial $(4 +5x^3...,4,This polynomial is not written in standard for...,2,True,Step 1: Identify what is being asked.\n\nThe p...,4
3,Evaluate $\left\lceil3\left(6-\frac12\right)\r...,17,"Firstly, $3\left(6-\frac12\right)=18-1-\frac12...",3,True,Step 1: Identify the expression to evaluate.\n...,17
4,Sam is hired for a 20-day period. On days that...,6,Call $x$ the number of days Sam works and $y$ ...,4,True,Let \(d\) be the number of days Sam did not wo...,6


In [13]:
df.to_parquet(
    "../../data/out/distillation/hendrycks_math_distilled_extra_detailed_final.parquet", index=False
)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")